# Precision: repeatability and intermediate precision (ISO 5725)

**Question:** how much do results on the same sample scatter *within a day*, and how
much more *across days*? And how far apart may two duplicate results be before the
lab should suspect a problem?

- **Repeatability** (s<sub>r</sub>): same sample, same analyst, same instrument, same
  day.
- **Intermediate precision** (s<sub>I</sub>): same lab and method, but over days, with
  the changes routine work brings: new calibrations, reagent lots, analysts.

Chapter 1 defines both conditions. This notebook estimates them from one simple
design and the one-way analysis of variance (ANOVA) of ISO 5725.

> **Data and assumptions.** All data are synthetic, made by `scripts/make_data.py`:
> a river-water sample at about 20 µg/L Cu, analysed by ICP-OES in triplicate (three
> separate preparations) on each of 10 working days. Analysts A and B alternate by
> day. The sample is assumed homogeneous and stable over the two weeks. No real
> laboratory data are used.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from labqc.validation import intermediate_precision

DATA = Path("..") / "data" / "synthetic"
if not DATA.exists():  # run from the repository root
    DATA = Path("data") / "synthetic"

prec = pd.read_csv(DATA / "precision.csv", parse_dates=["date"])

# One quiet chart style for the handbook: one series colour, grey references.
BLUE, GREY, INK = "#2a78d6", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
})

n_days, n_reps = prec["day"].nunique(), prec.groupby("day").size().iloc[0]
print(f"{n_days} days x {n_reps} replicates = {len(prec)} results, "
      f"{prec['date'].min():%d %b} to {prec['date'].max():%d %b %Y}")

In [ ]:
day_means = prec.groupby("date")["cu_ug_L"].mean()
grand = prec["cu_ug_L"].mean()
half_day = pd.Timedelta(hours=8)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.scatter(prec["date"], prec["cu_ug_L"], s=22, color=BLUE, zorder=3)
ax.hlines(day_means, day_means.index - half_day, day_means.index + half_day,
          color=INK, lw=1.5)
ax.axhline(grand, color=GREY, ls="--", lw=1)
ax.annotate(f"grand mean {grand:.2f}", (1.0, grand),
            xycoords=("axes fraction", "data"), xytext=(4, 0),
            textcoords="offset points", va="center", fontsize=9, color=INK)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_ylabel("Cu (µg/L)")
ax.set_title("One sample, three replicates a day for 10 days (line = day mean)")
plt.tight_layout()
plt.show()

Two kinds of scatter are visible: the three points within each day, and the day
means moving up and down. The ANOVA separates them.

## Why not just take a standard deviation?

In [ ]:
per_day_sd = prec.groupby("day")["cu_ug_L"].std(ddof=1)
print(f"SD of all 30 results:      {prec['cu_ug_L'].std(ddof=1):.2f} µg/L")
lo, hi = per_day_sd.min(), per_day_sd.max()
print(f"SD within one day (n = 3): {lo:.2f} to {hi:.2f} µg/L")

Neither answers the question. The SD of all 30 results mixes the two kinds of
scatter in unknown proportions. A single day's triplicate is far too small: the
same method gives anything from 0.03 to 0.68 µg/L depending on the day. A
validation report that quotes one day's SD as "the precision" can be more than
ten times too optimistic.

## The one-way ANOVA

With *p* days and *n* replicates per day, ISO 5725-2 estimates:

$$
s_r^2 = \text{MS}_\text{within}, \qquad
s_\text{between}^2 = \frac{\text{MS}_\text{between} - \text{MS}_\text{within}}{n}, \qquad
s_I^2 = s_r^2 + s_\text{between}^2
$$

MS<sub>within</sub> is the pooled variance of the replicates about their day mean;
MS<sub>between</sub> is *n* times the variance of the day means. If the
difference is negative, s<sub>between</sub> is set to zero.

In [ ]:
ip = intermediate_precision(prec, value_col="cu_ug_L", group_col="day")

display(pd.DataFrame(
    {
        "SD (µg/L)": [ip.sd_repeatability, ip.sd_between, ip.sd_intermediate],
        "RSD (%)": [ip.rsd_repeatability_percent, 100 * ip.sd_between / ip.mean,
                    ip.rsd_intermediate_percent],
    },
    index=["repeatability, s_r", "between days", "intermediate precision, s_I"],
).round(2))
print(f"Mean {ip.mean:.2f} µg/L; F = {ip.f:.2f}, p = {ip.p_value:.3f}")

### Check with statsmodels

The same ANOVA from a general-purpose statistics package, so the numbers do not
rest on `labqc` alone:

In [ ]:
model = smf.ols("cu_ug_L ~ C(day)", data=prec).fit()
table = sm.stats.anova_lm(model)
display(table.round(4))

assert np.isclose(table.loc["Residual", "mean_sq"], ip.ms_within)
assert np.isclose(table.loc["C(day)", "mean_sq"], ip.ms_between)
print("labqc and statsmodels agree on both mean squares.")

## Reading the result

**Repeatability is about 2.0 % and intermediate precision about 2.4 %** at
20 µg/L. The day-to-day component is real but smaller than the within-day scatter.

**The F test is not significant (p ≈ 0.08).** That does *not* mean the days are
the same. With 10 days the test has little power to detect a between-day SD of
this size. ISO 5725 keeps the estimate of s<sub>between</sub> whatever the test
says; setting it to zero because p > 0.05 would understate s<sub>I</sub> and,
later, the measurement uncertainty.

**It agrees with the QC data.** The Nordtest notebook found u(R<sub>w</sub>) = 2.7 %
from five months of control results at 50 µg/L. A two-week study is expected to
show slightly less variation than five months of routine work, and it does. If the
two disagreed badly, one of them would not represent routine conditions.

## Using it: limits for duplicates

The difference between two results has a standard deviation of √2 × s. At about
95 % confidence, two results should differ by no more than (ISO 5725-6):

$$
r = 2.8\,s_r \quad \text{(same day)}, \qquad R_I = 2.8\,s_I \quad \text{(different days)}
$$

The factor 2.8 is 1.96 × √2.

In [ ]:
r_limit = 2.8 * ip.sd_repeatability
ri_limit = 2.8 * ip.sd_intermediate
print(f"Repeatability limit r  = {r_limit:.2f} µg/L ({100 * r_limit / ip.mean:.1f} %)")
print(f"Intermediate limit R_I = {ri_limit:.2f} µg/L "
      f"({100 * ri_limit / ip.mean:.1f} %)")

ranges = prec.groupby("day")["cu_ug_L"].agg(lambda v: v.max() - v.min())
critical = 3.3 * ip.sd_repeatability
print(f"Largest same-day range of three = {ranges.max():.2f} µg/L "
      f"(day {ranges.idxmax()}); critical range 3.3 s_r = {critical:.2f} µg/L")

So two duplicates of a sample near 20 µg/L, analysed in the same run, should agree
within about 1.1 µg/L (5.7 %). A larger difference is a reason to re-analyse before
reporting. Because the limit is relative, it can be used across the working range
as long as precision is roughly proportional to concentration; near the LOQ it is
not, and an absolute limit is needed there.

For three replicates, ISO 5725-6 uses a critical range of 3.3 s<sub>r</sub> instead
of 2.8 s<sub>r</sub>; every day in this study is within it.

> **From the lab.** *[To write: how duplicate limits were set and used in a lab you
> worked in, and what happened when duplicates failed.]*

## References

- ISO 5725-2, basic method for repeatability and reproducibility.
- ISO 5725-3, intermediate measures of precision.
- ISO 5725-6, use of accuracy values in practice (repeatability limit, critical range).
- Eurachem, *The Fitness for Purpose of Analytical Methods*, section on precision.
- `labqc.validation.intermediate_precision`:
  [API documentation](https://mantas123456.github.io/lab-quality-toolkit/api/validation/).